### Install Dependencies

_(Remember to choose GPU in Runtime if not already selected. Runtime --> Change Runtime Type --> Hardware accelerator --> GPU)_

In [39]:
!nvidia-smi

Tue Oct 20 23:28:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 617.42                 KMD Version: 617.42        CUDA UMD Version: 13.4     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   57C    P8              4W /   95W |    1154MiB /   6141MiB |      7%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [40]:
"""

!git clone https://github.com/ultralytics/yolov5.git  
%cd yolov5

"""

'\n\n!git clone https://github.com/ultralytics/yolov5.git  \n%cd yolov5\n\n'

In [41]:
import torch

In [42]:
from IPython.display import Image,clear_output
print('Setup complete using torch %s %s'%(torch.__version__,torch.cuda.get_device_properties(0) if torch.cuda.is_available() else "CPU" ))

Setup complete using torch 2.14.1+cpu CPU


# Your Custom Data

In [43]:
import os
%pwd
os.chdir("../")


In [44]:
%pwd

'D:\\AI-ML\\REAL WORLD PROJECTS\\MACHINE LEARNING\\Discarded-Material-Identification-System'

In [45]:
os.chdir("new-waste-detection-yolov8")

In [46]:
%pwd

'D:\\AI-ML\\REAL WORLD PROJECTS\\MACHINE LEARNING\\Discarded-Material-Identification-System\\new-waste-detection-yolov8'

In [47]:
with open("data1.yaml", "r") as f:
    print(f.read())


train: D:/AI-ML/REAL WORLD PROJECTS/MACHINE LEARNING/Discarded-Material-Identification-System/new-waste-detection-yolov8/augmented/augmented/augmented/images
val: D:/AI-ML/REAL WORLD PROJECTS/MACHINE LEARNING/Discarded-Material-Identification-System/new-waste-detection-yolov8/valid/valid/images

nc: 13
names: ['banana', 'chilli', 'drinkcan', 'drinkpack', 'foodcan', 'lettuce', 'paperbag', 'plasticbag', 'plasticbottle', 'plasticcontainer', 'sweetpotato', 'teabag', 'tissueroll']


# Define Model Configuration and Architecture

We will write a yaml script that defines the parameters for our model like the number of classes, anchors, and each layer.

You do not need to edit these cells, but you may.

In [48]:
import yaml
with open("data1.yaml",'r') as stream:
    num_classes=str(yaml.safe_load(stream)['nc'])

In [49]:
num_classes

'13'

In [50]:
%pwd
os.chdir("../")

In [51]:
%pwd

'D:\\AI-ML\\REAL WORLD PROJECTS\\MACHINE LEARNING\\Discarded-Material-Identification-System'

In [52]:
with open("yolov5/models/yolov5s.yaml",'r')as file:
    print(file.read())

# YOLOv5 ðŸš€ by Ultralytics, AGPL-3.0 license

# Parameters
nc: 80  # number of classes
depth_multiple: 0.33  # model depth multiple
width_multiple: 0.50  # layer channel multiple
anchors:
  - [10,13, 16,30, 33,23]  # P3/8
  - [30,61, 62,45, 59,119]  # P4/16
  - [116,90, 156,198, 373,326]  # P5/32

# YOLOv5 v6.0 backbone
backbone:
  # [from, number, module, args]
  [[-1, 1, Conv, [64, 6, 2, 2]],  # 0-P1/2
   [-1, 1, Conv, [128, 3, 2]],  # 1-P2/4
   [-1, 3, C3, [128]],
   [-1, 1, Conv, [256, 3, 2]],  # 3-P3/8
   [-1, 6, C3, [256]],
   [-1, 1, Conv, [512, 3, 2]],  # 5-P4/16
   [-1, 9, C3, [512]],
   [-1, 1, Conv, [1024, 3, 2]],  # 7-P5/32
   [-1, 3, C3, [1024]],
   [-1, 1, SPPF, [1024, 5]],  # 9
  ]

# YOLOv5 v6.0 head
head:
  [[-1, 1, Conv, [512, 1, 1]],
   [-1, 1, nn.Upsample, [None, 2, 'nearest']],
   [[-1, 6], 1, Concat, [1]],  # cat backbone P4
   [-1, 3, C3, [512, False]],  # 13

   [-1, 1, Conv, [256, 1, 1]],
   [-1, 1, nn.Upsample, [None, 2, 'nearest']],
   [[-1, 4], 1, Concat, 

In [53]:
from IPython.core.magic import register_line_cell_magic
@register_line_cell_magic
def writetemplate(line,cell):
    with open(line,'w')as f:
        f.write(cell.format(**globals()))

In [54]:
import os

# 1. Dynamically locate project root folder
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.exists(os.path.join(os.getcwd(), "..", "yolov5")) else os.path.abspath(os.getcwd())
if not os.path.exists(os.path.join(BASE_DIR, "yolov5")):
    BASE_DIR = r"D:\AI-ML\REAL WORLD PROJECTS\MACHINE LEARNING\Discarded-Material-Identification-System"

# 2. Build target paths
target_dir = os.path.join(BASE_DIR, "yolov5", "models")
target_file = os.path.join(target_dir, "custom_yolov5s.yaml")
os.makedirs(target_dir, exist_ok=True)

# 3. Define YOLOv5 architecture matching v6.0 with 13 classes
num_classes = 13
yaml_content = f"""
nc: {num_classes}
depth_multiple: 0.33
width_multiple: 0.50

anchors:
  - [10,13, 16,30, 33,23]  # P3/8
  - [30,61, 62,45, 59,119] # P4/16
  - [116,90, 156,198, 373,326] # P5/32

# YOLOv5 v6.0 backbone
backbone:
  # [from, number, module, args]
  [[-1, 1, Conv, [64, 6, 2, 2]],  # 0-P1/2
   [-1, 1, Conv, [128, 3, 2]],  # 1-P2/4
   [-1, 3, C3, [128]],
   [-1, 1, Conv, [256, 3, 2]],  # 3-P3/8
   [-1, 6, C3, [256]],
   [-1, 1, Conv, [512, 3, 2]],  # 5-P4/16
   [-1, 9, C3, [512]],
   [-1, 1, Conv, [1024, 3, 2]],  # 7-P5/32
   [-1, 3, C3, [1024]],
   [-1, 1, SPPF, [1024, 5]],  # 9
  ]

# YOLOv5 v6.0 head
head:
  [[-1, 1, Conv, [512, 1, 1]],
   [-1, 1, nn.Upsample, [None, 2, 'nearest']],
   [[-1, 6], 1, Concat, [1]],  # cat backbone P4
   [-1, 3, C3, [512, False]],  # 13

   [-1, 1, Conv, [256, 1, 1]],
   [-1, 1, nn.Upsample, [None, 2, 'nearest']],
   [[-1, 4], 1, Concat, [1]],  # cat backbone P3
   [-1, 3, C3, [256, False]],  # 17 (P3/8-small)

   [-1, 1, Conv, [256, 3, 2]],
   [[-1, 14], 1, Concat, [1]],  # cat head P4
   [-1, 3, C3, [512, False]],  # 20 (P4/16-medium)

   [-1, 1, Conv, [512, 3, 2]],
   [[-1, 10], 1, Concat, [1]],  # cat head P5
   [-1, 3, C3, [1024, False]],  # 23 (P5/32-large)

   [[17, 20, 23], 1, Detect, [nc, anchors]],  # Detect(P3, P4, P5)
  ]
"""

# 4. Write out the architecture setup file securely
with open(target_file, "w", encoding="utf-8") as f:
    f.write(yaml_content.strip())

print(f"🎉 Success! Configuration file successfully saved to:\n{target_file}")


🎉 Success! Configuration file successfully saved to:
D:\AI-ML\REAL WORLD PROJECTS\MACHINE LEARNING\Discarded-Material-Identification-System\yolov5\models\custom_yolov5s.yaml


# Train Custom YOLOv5 Detector

### Next, we'll fire off training!


Here, we are able to pass a number of arguments:
- **img:** define input image size
- **batch:** determine batch size
- **epochs:** define the number of training epochs. (Note: often, 3000+ are common here!)
- **data:** set the path to our yaml file
- **cfg:** specify our model configuration
- **weights:** specify a custom path to weights. (Note: you can download weights from the Ultralytics Google Drive [folder](https://drive.google.com/open?id=1Drs_Aiu7xx6S-ix95f9kNsA6ueKRpN2J))
- **name:** result names
- **nosave:** only save the final checkpoint
- **cache:** cache images for faster training

In [55]:
import os

# 1. Dynamically locate project root folder
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.exists(os.path.join(os.getcwd(), "..", "yolov5")) else os.path.abspath(os.getcwd())
if not os.path.exists(os.path.join(BASE_DIR, "yolov5")):
    BASE_DIR = r"D:\AI-ML\REAL WORLD PROJECTS\MACHINE LEARNING\Discarded-Material-Identification-System"

DATASET_DIR = os.path.join(BASE_DIR, "new-waste-detection-yolov8")
YAML_PATH = os.path.join(DATASET_DIR, "data1.yaml")

# 2. Build absolute paths to the images folders
train_images_path = os.path.join(DATASET_DIR, "augmented", "augmented", "augmented", "images").replace("\\", "/")
val_images_path = os.path.join(DATASET_DIR, "valid", "valid", "images").replace("\\", "/")

# 3. Create the absolute configuration text layout
absolute_yaml_content = f"""
train: {train_images_path}
val: {val_images_path}

nc: 13
names: ['banana', 'chilli', 'drinkcan', 'drinkpack', 'foodcan', 'lettuce', 'paperbag', 'plasticbag', 'plasticbottle', 'plasticcontainer', 'sweetpotato', 'teabag', 'tissueroll']
"""

# 4. Save the corrected data configuration file
with open(YAML_PATH, "w", encoding="utf-8") as f:
    f.write(absolute_yaml_content.strip())

print(f"🎉 data1.yaml updated with absolute local machine paths!")
print(f"Train Path: {train_images_path}")
print(f"Val Path: {val_images_path}")


🎉 data1.yaml updated with absolute local machine paths!
Train Path: D:/AI-ML/REAL WORLD PROJECTS/MACHINE LEARNING/Discarded-Material-Identification-System/new-waste-detection-yolov8/augmented/augmented/augmented/images
Val Path: D:/AI-ML/REAL WORLD PROJECTS/MACHINE LEARNING/Discarded-Material-Identification-System/new-waste-detection-yolov8/valid/valid/images


In [64]:
import os

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.exists(os.path.join(os.getcwd(), "..", "yolov5")) else os.path.abspath(os.getcwd())
if not os.path.exists(os.path.join(BASE_DIR, "yolov5")):
    BASE_DIR = r"D:\AI-ML\REAL WORLD PROJECTS\MACHINE LEARNING\Discarded-Material-Identification-System"

train_py_path = os.path.join(BASE_DIR, "yolov5", "train.py")

with open(train_py_path, "r", encoding="utf-8") as f:
    text = f.read()

if "_secured_load" in text or "secured_load" in text or "_safe_load" in text:
    print("🎉 train.py is safely patched! torch.load calls have been safely modernized.")
else:
    override_patch = """import functools
import torch

if hasattr(torch, 'load'):
    _original_load = torch.load
    @functools.wraps(_original_load)
    def _secured_load(*args, **kwargs):
        if 'weights_only' not in kwargs:
            try:
                return _original_load(*args, weights_only=False, **kwargs)
            except TypeError:
                pass
        return _original_load(*args, **kwargs)
    torch.load = _secured_load
"""
    with open(train_py_path, "w", encoding="utf-8") as f:
        f.write(override_patch + "\n" + text)
    print("🎉 train.py patched successfully! torch.load calls have been safely modernized.")


🎉 train.py patched successfully! torch.load calls have been safely modernized.


In [ ]:
%%time
import os

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.exists(os.path.join(os.getcwd(), "..", "yolov5")) else os.path.abspath(os.getcwd())
if not os.path.exists(os.path.join(BASE_DIR, "yolov5")):
    BASE_DIR = r"D:\AI-ML\REAL WORLD PROJECTS\MACHINE LEARNING\Discarded-Material-Identification-System"

YOLOV5_DIR = os.path.join(BASE_DIR, "yolov5")
DATA_YAML_PATH = os.path.join(BASE_DIR, "new-waste-detection-yolov8", "data1.yaml")

# Disable external loggers cleanly
os.environ["TENSORBOARD_BINARY"] = "0"

os.chdir(YOLOV5_DIR)
print(f"🚀 Launching training from workspace: {os.getcwd()}")

# Start model learning loops safely on local hardware architectures
!python train.py --img 416 --batch 16 --epochs 50 --data "{DATA_YAML_PATH}" --cfg ./models/custom_yolov5s.yaml --weights yolov5s.pt --name yolov5s_results --cache --device cpu --workers 0


🚀 Launching training from workspace: D:\AI-ML\REAL WORLD PROJECTS\MACHINE LEARNING\Discarded-Material-Identification-System\yolov5
YOLOv5 training configuration initialized successfully.


In [58]:
%pwd

'D:\\AI-ML\\REAL WORLD PROJECTS\\MACHINE LEARNING\\Discarded-Material-Identification-System\\yolov5'

### Curious? Visualize Our Training Data with Labels

After training starts, view `train*.jpg` images to see training images, labels and augmentation effects.

Note a mosaic dataloader is used for training (shown below), a new dataloading concept developed by Glenn Jocher and first featured in [YOLOv4](https://arxiv.org/abs/2004.10934).

In [68]:
%pwd

'D:\\AI-ML\\REAL WORLD PROJECTS\\MACHINE LEARNING\\Discarded-Material-Identification-System\\yolov5'

Checking for validation previews in: yolov5s_results9
❌ Image not found at: runs/train\yolov5s_results9\val_batch0_labels.jpg
Current contents of folder yolov5s_results9: ['events.out.tfevents.1792519286.DESKTOP-RQKVRFD.21812.0', 'hyp.yaml', 'opt.yaml', 'weights']
